# Spur Solver Z3 Optimize Semantics Design

**Status:** Approved for implementation from the 2026-08-18 audit  
**Design epic:** `bd-b37`  
**Scope:** `crates/spur-solver`

## Goal

Make the typed solver API a faithful, observable adapter for Z3 Optimize/νZ: weighted soft constraints must retain aggregate MaxSMT semantics, objective bounds must distinguish finite, infinite, and strict optima, and lex/Pareto/box modes must return results matching Z3's execution model.

## Non-goals

- Embedding or linking Z3 into the Rust process.
- Replacing the fixed-argv `z3 -in` process boundary.
- Adding proof certificates or combining Optimize with unsat-core production.
- Claiming NS-Mermaid can prove optimization optimality; its formal cells here cover protocol order and result classification only.


## Contract decisions

### 1. Soft identity and grouping

`ConstraintDecl.id` is diagnostic identity. It stays unique across constraints and is returned for violated soft constraints; it is never encoded as Z3 `:id`.

A new optional `ConstraintDecl.group` is valid only for soft constraints. Group names may repeat. The encoder maps only `group` to `assert-soft :id`. Omitting `group` omits the SMT tag, so every ungrouped soft constraint participates in the same aggregate weighted objective.

### 2. Optimization-active protocol

Optimization is active when the request has at least one soft constraint or explicit objective. In that case the encoder always emits `:opt.priority`.

Each satisfiable optimization cycle emits, in order:

1. `check-sat`
2. `get-objectives`
3. one combined `get-value` containing declared variables, every soft expression, and every explicit objective expression

Lex uses one cycle. Box uses one cycle per generated Z3 objective followed by a terminal probe. Pareto enumerates at most `max_solutions` cycles followed by a terminal probe. `max_solutions` defaults to 16 and is capped at 64; the pre-solve artifact `sol_5d4d2816017d4af8` found 68 nominal 15 KiB solution envelopes fit below the existing 1 MiB stdout cap, so 64 is the conservative wire limit.

### 3. Typed result envelope

The existing top-level `model` remains the first returned model for backward compatibility. Optimization requests additionally return:

- `OptimizationResult { priority, solutions, termination }`
- `OptimizationTermination = complete | solution_limit | unknown`
- Per-solution model, explicit objective value/bound pairs, per-soft-constraint satisfaction, and per-group satisfied/violated weights
- `ObjectiveBound = finite { exact } | infinite { exact } | strict { exact }`

`exact` preserves Z3 arithmetic text losslessly. A bound containing `oo` is infinite; otherwise a bound containing `epsilon` is strict; all other bounds are finite. The formal classifier below requires those cases to be total and mutually exclusive.

For Pareto or box, the terminal `unsat` means enumeration completed and does not replace the top-level satisfiable status. If Pareto's probe is `sat`, termination is `solution_limit`. An `unknown` after at least one point produces a partial satisfiable result with termination `unknown`; an initial `unknown` remains a top-level unknown result.

### 4. Runtime compatibility

The response includes the probed Z3 version. Production binaries older than Z3 4.8.12 are rejected as unavailable before submitting an optimization script; injected fake runners with an unknown version remain supported for deterministic tests.


In [ ]:
sequenceDiagram
    participant Client
    participant Encoder
    participant Z3
    participant Parser
    Note over Client,Parser: @spec Z3OPT-PROTOCOL
    Client->>Encoder: typed_request
    Note over Client,Encoder: @message REQUEST<br/>@from Client<br/>@to Encoder<br/>@event typed_request<br/>@order 1<br/>@when true<br/>@ensures REQUEST_ACCEPTED: true
    Encoder->>Z3: optimization_script
    Note over Encoder,Z3: @message SCRIPT<br/>@from Encoder<br/>@to Z3<br/>@event optimization_script<br/>@order 2<br/>@when true<br/>@ensures SCRIPT_COMPLETE: true
    Z3-->>Parser: solve_status
    Note over Z3,Parser: @message STATUS<br/>@from Z3<br/>@to Parser<br/>@event solve_status<br/>@order 3<br/>@when true<br/>@ensures STATUS_FIRST: true
    Z3-->>Parser: objective_bounds
    Note over Z3,Parser: @message BOUNDS<br/>@from Z3<br/>@to Parser<br/>@event objective_bounds<br/>@order 4<br/>@when true<br/>@ensures BOUNDS_AFTER_STATUS: true
    Z3-->>Parser: model_values
    Note over Z3,Parser: @message VALUES<br/>@from Z3<br/>@to Parser<br/>@event model_values<br/>@order 5<br/>@when true<br/>@ensures VALUES_AFTER_BOUNDS: true
    Parser-->>Client: typed_response
    Note over Parser,Client: @message RESPONSE<br/>@from Parser<br/>@to Client<br/>@event typed_response<br/>@order 6<br/>@when true<br/>@ensures RESPONSE_LAST: true
    Note over Client,Parser: @verify PROTOCOL: prove sequence_protocol

In [ ]:
flowchart TD
    SPEC["`@spec OBJECTIVE-BOUND-CLASSIFIER
@type Status = enum[infinite, strict, finite]
@input has_oo: Bool
@input has_epsilon: Bool
@output status: Status
@requires PRE: true`"]
    INFINITE["`@branch INFINITE
@when has_oo = true
@ensures INFINITE_STATUS: status = infinite`"]
    STRICT["`@branch STRICT
@when has_oo = false and has_epsilon = true
@ensures STRICT_STATUS: status = strict`"]
    FINITE["`@branch FINITE
@when has_oo = false and has_epsilon = false
@ensures FINITE_STATUS: status = finite`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> INFINITE --> CHECK
    SPEC --> STRICT --> CHECK
    SPEC --> FINITE --> CHECK

## Parsing, compatibility, and verification

### Parser rules

The S-expression parser consumes command responses positionally, not by searching for the first list-shaped form. A sat cycle must contain exactly one `objectives` form followed by exactly one combined `get-value` form. The parser validates declared-variable symbols, then consumes soft and objective values in request order.

For typed requests, Z3 objective entries are matched to soft groups in first-declaration order, then explicit objectives in request order. A malformed count, duplicate variable binding, missing cycle payload, unexpected extra form, or inconsistent terminal status remains a structured parse error.

Raw SMT admits `get-objectives`. When present, its bounds are exposed through the same optimization envelope with unknown priority/direction metadata rather than discarded; ordinary raw SAT/model behavior remains unchanged.

### Compatibility and migration

- Existing request JSON remains valid through serde defaults for `group` and `max_solutions`.
- Existing response consumers retain top-level `status` and `model`; `optimization` and `solver_version` are additive.
- Hard named constraints continue to use `:named id` and unsat cores.
- Soft `id` no longer changes objective grouping. Callers that intentionally relied on unique lexicographic soft groups must move those names to `group`.
- Cache fingerprints and persisted artifacts include all new request/result fields.

### RED/GREEN verification matrix

1. Encoder RED: two uniquely named, ungrouped soft constraints must emit no `:id`; repeated `group` values must emit the same Z3 `:id`.
2. Real-Z3 RED: conflicting weights 1 and 100 with unique diagnostic IDs must choose the weight-100 constraint.
3. Encoder RED: soft-only Pareto/box requests must emit `:opt.priority`.
4. Parser RED: finite, `oo`, negative-`oo`, and epsilon bounds classify losslessly.
5. Real-Z3 RED: unconstrained Real maximization reports an infinite bound; `x < 1` reports a strict bound.
6. Real-Z3 RED: Pareto enumerates the complete small frontier; a lower `max_solutions` reports `solution_limit`.
7. Real-Z3 RED: box returns one model per independent objective and preserves each independent bound.
8. Raw-gate RED: `get-objectives` is accepted and parsed.
9. Version RED: production version parsing accepts 4.8.12+ and rejects older parseable versions; unknown injected runners remain testable.
10. Documentation: the real-Z3 command includes `-- --ignored`, and the deferred-feature list matches shipped behavior.

All Rust commands use `scripts/spur-cargo`; real-Z3 tests use `SPUR_REMOTE=0 SPUR_TEST_Z3=1 ... -- --ignored`.
